# EDA — Panel núcleo de rendimientos agrícolas

Análisis exploratorio de `data/processed/panel_nucleo.parquet` (merge puro de 4 fuentes, sin feature engineering).

**Objetivo del proyecto:** pronosticar el rendimiento (`rinde_kgha`) y detectar campañas anómalas en la región núcleo (26 departamentos × 2 cultivos × 44 campañas, 1981/82–2024/25).

**Estructura del EDA:**
1. Estructura y granularidad del panel
2. Valores faltantes (NDVI desde 2002, radiación temprana)
3. Variable objetivo: `rinde_kgha`
4. MAGyP — superficie y pérdida de cosecha
5. ONI / ENSO (El Niño–La Niña)
6. NASA POWER — clima
7. NDVI (subconjunto 2002+)
8. Correlaciones globales y teleconexión
9. Síntesis para el modelado

> **Nota sobre NaN:** no se eliminan filas. El NDVI no existe físicamente antes de 2002 (MODIS) y la radiación solar tiene huecos en campañas tempranas. Cada análisis maneja los faltantes localmente (subset temporal o correlaciones por pares), nunca con un `dropna` global.

> Soja (~2.700 kg/ha) y maíz (~6.700 kg/ha) están en escalas muy distintas: **todo se analiza separado por cultivo o sobre el rinde normalizado.**

## Splits temporales

El ETL no asigna splits — se definen aquí para poder cambiar los cortes sin re-correr el pipeline.

| Split | Campañas | Función |
|-------|----------|---------|
| **train** | 1981/82 – 2017/18 | Ajuste de modelos y normalización |
| **val** | 2018/19 – 2020/21 | Tuning de hiperparámetros |
| **test** | 2021/22 – 2024/25 | Evaluación final (incluye sequía 2022/23) |

> **Importante:** cualquier estadística de normalización (media, desvío) debe computarse **solo sobre train**, nunca sobre el panel completo.

## 1. Estructura y granularidad del panel

Granularidad: **departamento × campaña × cultivo**. Verificamos que la clave sea única, los rangos del split temporal y qué tan balanceado está el panel (¿hay departamentos sin observación en alguna campaña?).

## 2. Valores faltantes

El target y la mayoría de las features tienen cobertura casi completa. Los faltantes se concentran en dos bloques estructurales (no son errores de descarga):

- **NDVI**: ~52% NaN — MODIS opera recién desde **2002**.
- **Radiación solar (`allsky_*`)**: ~7% NaN, concentrado en campañas tempranas.
- Resto de NASA (temp/precip/humedad/viento): ~2% en algunos meses de verano.

El heatmap temporal muestra *cuándo* falta cada cosa, lo que justifica tratar a NDVI como un ablation sobre el subset 2002+.

## 3. Variable objetivo: `rinde_kgha`

El target. Lo miramos desde varios ángulos relevantes para el proyecto:
- **distribución** por cultivo (escalas muy distintas),
- **evolución temporal** — hay tendencia tecnológica creciente que conviene separar de la variabilidad climática,
- **heterogeneidad espacial** por departamento,
- **distribution shift** entre train/val/test (clave porque el split es temporal),
- **anomalías**: residuo del rinde respecto de su tendencia → base para la detección de campañas malas.

## 4. MAGyP — superficie y pérdida de cosecha

Además del rinde, MAGyP trae superficie sembrada/cosechada y producción. El cociente **cosechada/sembrada** es un proxy directo de pérdida: cuando cae fuerte, hubo abandono de lotes (otra señal de campaña anómala, complementaria al rinde).

## 5. ONI / ENSO (El Niño – La Niña)

El ONI es una serie **global** (igual para todos los deptos/cultivos de una campaña). Resumimos los 5 meses en un `_oni_mean` (Oct–Feb) y clasificamos cada campaña en La Niña (≤ −0.5), Neutro o El Niño (≥ +0.5). Es el principal driver climático *conocido de antemano*, así que su relación con el rinde es central para el pronóstico estacional.

> Para aislar el efecto climático de la tendencia tecnológica, el ENSO se compara contra el **residuo de rinde** (`_rinde_resid_pct`) calculado en la sección 3.

## 6. NASA POWER — clima

49 columnas (7 variables × 7 meses sep–mar). Miramos primero el **perfil estacional** (para ubicar la ventana crítica del cultivo), luego la **correlación de los meses críticos con el rinde**, y finalmente la precipitación de verano contra la anomalía de rinde, coloreada por fase ENSO.

## 7. NDVI — índice de vegetación (subset 2002+)

MODIS opera desde 2002, así que el NDVI se analiza **solo sobre el subconjunto 2002–2024** (sin imputar lo que no existe). Es un proxy directo del vigor del cultivo, por lo que esperamos correlación positiva con el rinde y caídas en campañas secas.

## 8. Correlaciones globales y teleconexión

Ranking de la correlación de cada feature candidata con el rinde (por cultivo) y verificación del mecanismo ENSO → precipitación → rinde.

## 9. Síntesis — hallazgos para el modelado

**Estructura**
- Panel casi balanceado: 26 deptos × 2 cultivos × 44 campañas (soja arranca con 25 deptos en alguna campaña temprana). Clave única, sin duplicados.
- Split temporal: train 1981–2017, val 2018–2020, test 2021–2023. El test contiene **2022/23 (sequía extrema)** → es un test exigente y realista.

**Target (`rinde_kgha`)**
- Soja y maíz en escalas distintas → modelar por separado o con el cultivo como feature + normalización.
- Fuerte **tendencia tecnológica** creciente: conviene incluir el año/tendencia como feature o trabajar sobre el residuo detrend para que el modelo capture señal climática y no solo el año.
- **Distribution shift** train→test: el rinde medio sube en val/test (tendencia). Las métricas absolutas no son comparables entre splits sin tener esto en cuenta.

**Señales predictivas (correlación con rinde)**
- **Tmax de enero**: la correlación más fuerte y negativa (≈ −0.4) → estrés térmico en llenado de grano.
- **Humedad y precipitación de verano (dic–feb)**: positivas → agua en período crítico.
- **ENSO**: La Niña deprime el rinde (−7 a −11% vs tendencia), El Niño lo favorece (+7 a +9%). ONI es un predictor estacional disponible *antes* de la campaña → útil para pronóstico.
- **NDVI** (solo 2002+): correlación moderada (≈ 0.29) con rinde.

**Faltantes**
- NDVI 2002+ → usarlo como **ablation** sobre el subset, no en el modelo base 1981–2024.
- Radiación solar con huecos tempranos → imputar o excluir según el modelo.

**Anomalías**
- El residuo del rinde vs tendencia identifica limpiamente las campañas malas (2022, 1988, 2008), que coinciden con La Niña y caídas de % cosechado → base sólida para el componente de detección de anomalías.